# 0x8dxd 按市场(question)聚合：买卖 Yes/No、总 PnL、回撤、Sharpe

数据源：`trades_cleaned.csv`
- 按 **question** 聚合，每市场：buy_yes / buy_no / sell_yes / sell_no 笔数
- 总 PnL（净现金流：卖出收入 - 买入成本）
- 回撤（按时间序累计 PnL 的最大回撤）
- Sharpe（按单笔交易收益序列年化近似）

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# 数据路径（相对项目根或改成本机路径）
DATA_DIR = Path("eda/01_etl/output/0x8dxd_full")
df = pd.read_csv(DATA_DIR / "trades_cleaned.csv")
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce").astype("Int64")
df["size"] = pd.to_numeric(df["size"], errors="coerce")
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["question", "side", "size", "price"])
print(f"共 {len(df)} 条成交，{df['question'].nunique()} 个市场")
df.head(3)

In [ ]:
# outcomeIndex: 0 视为 Yes 侧，1 视为 No 侧（二元市场）
df["is_yes"] = (df["outcomeIndex"].fillna(0).astype(int) == 0)
# 单笔现金流：买入为负，卖出为正
df["cashflow"] = np.where(df["side"] == "BUY", -df["size"] * df["price"], df["size"] * df["price"])

In [ ]:
# 按 question 聚合：buy_yes / buy_no / sell_yes / sell_no 笔数 + 总 PnL
total_pnl = df.groupby("question")["cashflow"].sum()
idx = total_pnl.index
buy_yes  = df[(df["side"] == "BUY") & df["is_yes"]].groupby("question").size().reindex(idx).fillna(0).astype(int)
buy_no   = df[(df["side"] == "BUY") & ~df["is_yes"]].groupby("question").size().reindex(idx).fillna(0).astype(int)
sell_yes = df[(df["side"] == "SELL") & df["is_yes"]].groupby("question").size().reindex(idx).fillna(0).astype(int)
sell_no  = df[(df["side"] == "SELL") & ~df["is_yes"]].groupby("question").size().reindex(idx).fillna(0).astype(int)

agg = pd.DataFrame({
    "buy_yes": buy_yes,
    "buy_no": buy_no,
    "sell_yes": sell_yes,
    "sell_no": sell_no,
    "total_pnl": total_pnl,
}).fillna(0)
agg = agg.astype({c: int for c in ["buy_yes","buy_no","sell_yes","sell_no"]})
agg.index.name = "question"
agg.head(10)

In [ ]:
# 按市场、按时间排序，算累计 PnL → 回撤、Sharpe
def market_drawdown_sharpe(g: pd.DataFrame) -> pd.Series:
    g = g.sort_values("timestamp")
    cf = g["cashflow"].values
    cum = np.cumsum(cf)
    peak = np.maximum.accumulate(cum)
    dd = peak - cum
    max_dd = np.max(dd) if len(dd) else 0.0
    # Sharpe: mean(cashflow)/std(cashflow)*sqrt(n)，std=0 时置 0
    n = len(cf)
    if n < 2 or np.std(cf) == 0:
        sharpe = 0.0
    else:
        sharpe = (np.mean(cf) / np.std(cf)) * np.sqrt(n)
    return pd.Series({"max_drawdown": max_dd, "sharpe": sharpe, "n_trades": n})

extra = df.groupby("question").apply(market_drawdown_sharpe)
agg = agg.join(extra)
agg.head(10)

In [ ]:
# 按总 PnL 排序，看表现最好/最差市场
agg_sorted = agg.sort_values("total_pnl", ascending=False).reset_index()
agg_sorted["market_category"] = agg_sorted["question"].map(
    df.drop_duplicates("question").set_index("question")["market_category"]
)
cols = ["question", "market_category", "buy_yes", "buy_no", "sell_yes", "sell_no", "total_pnl", "max_drawdown", "sharpe", "n_trades"]
display(agg_sorted[cols].head(20))

In [ ]:
# 按市场类型汇总
agg_sorted.groupby("market_category").agg(
    markets=("question", "count"),
    total_pnl=("total_pnl", "sum"),
    mean_sharpe=("sharpe", "mean"),
    mean_drawdown=("max_drawdown", "mean"),
).round(4)

In [ ]:
# 导出按市场聚合表（可选）
out_path = DATA_DIR / "markets_agg.csv"
agg_sorted[cols].to_csv(out_path, index=False, encoding="utf-8")
print(f"已写入 {out_path}")